# Paper Results: Simulation vs. Experiment

**Branch:** `paper/results`  
**Solver:** legacy `relax_to_steady`, no distributed leakage, no BE-CV or enthalpy fixes  
**Tests:** 0116, 0121 (17 ft chimney) | 0123, 0126 (15 ft chimney)  
**Cap:** rain_cap (Cp = -0.30, calibrated)  

Generates all plots + metrics for the Results section:
- D1: Representative temperature overlays (17 ft + 15 ft)
- D2: Representative pressure overlays (17 ft + 15 ft)
- D3: RMSE/Bias summary (temperature + pressure, cracked vs closed)
- D4: Mass flow time series + steady-state comparison
- D5: Height scaling summary (15 ft vs 17 ft)
- D6: TC radial correction check (appendix)
- Report cell with BEGIN/END delimiters

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import time as _time
import datetime
import json
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d

REPO_ROOT = Path('..')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'dev' / 'exp_data'))

from flued_module.chimney import Chimney
from flued_module.material_properties import MaterialProperties
from experiment_loader import ExperimentLoader

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 150

mat_props = MaterialProperties(str(REPO_ROOT / 'flued_module' / 'data' / 'pipe_props.json'))
el = ExperimentLoader(
    metadata_path=REPO_ROOT / 'dev' / 'exp_data' / 'experiments_metadata.json'
)

print('Setup complete')
print(f'Available tests: {el.test_ids}')

## Paper Results Config

In [ ]:
# ============================================================
# Test grouping
# ============================================================
TESTS_17FT    = ['0116', '0121']
TESTS_15FT    = ['0123', '0126']
TESTS_ALL     = TESTS_17FT + TESTS_15FT
REP_TEST_17FT = '0116'   # representative for figures
REP_TEST_15FT = '0123'

HEIGHT_MAP    = {'17ft': 17, '15ft': 15}   # chimney height in feet

# ============================================================
# Calibrated parameters (same as results_validation)
# ============================================================
P_ATM          = 98457    # Pa  (lab elevation)
A_LEAK_CM2     = 75.0     # cm^2  inlet restriction leakage
TC1_OFFSET_C   = 10.0     # degC  TC1 reads wall; gas is ~10 C hotter
DOOR_WIDTH_IN  = 15.25
DOOR_HEIGHT_IN =  9.875
CD             = 0.62
K_STOVE        = 0.0
DOOR_RAMP_S    =  5.0     # seconds for cosine ramp

# ============================================================
# Solver settings -- EXPLICIT: legacy config, no new fixes
# ============================================================
DT             =  1.0     # s
SIM_TIME_MAX   = 1200     # s
# leakage_config = None  (enforced in run_sim)
# gas_energy_scheme = 'relax_to_steady' (default on this branch)

# ============================================================
# Geometry
# ============================================================
DIAMETER_IN    = 6.0
DIAMETER_FT    = DIAMETER_IN / 12
DIAMETER_M     = DIAMETER_IN * 0.0254
PIPE_AREA_M2   = np.pi * (DIAMETER_M / 2)**2

FINE_TOP_IN    = 15       # top 15 inches split finely
FINE_TOP_SEGS  = 15       # 1 inch per segment

BLACK_PIPE_IN  = 73.5
TEE1_VERT_IN   = 12.0;  TEE1_HORIZ_IN  = 6.5
ELBOW_VERT_IN  =  7.5;  ELBOW_HORIZ_IN = 8.5
HORIZ_IN       = 23.25
TEE2_VERT_IN   = 12.0;  TEE2_HORIZ_IN  = 6.5

# ============================================================
# Sensor map: (part_idx, height_in_within_part)
# Parts: 0=black_pipe, 1=horizontal, 2=bulk_vertical, 3=fine_top
# ============================================================
SENSOR_MAP = {
    '15ft': {
        'TC1': (0, 15.0),
        'TC2': (2, 12.0),
        'TC3': (2, 42.0),
        'TOP': (3,  3.0),
    },
    '17ft': {
        'TC1': (0, 15.0),
        'TC6': (2,  3.0),
        'TC2': (2, 34.75),
        'TC3': (2, 64.75),
        'TOP': (3,  3.0),
    },
}
TEMP_SENSORS   = {'15ft': ['TC1', 'TC2', 'TC3'],
                  '17ft': ['TC1', 'TC6', 'TC2', 'TC3']}
PRESS_SENSORS  = {'15ft': ['TC1', 'TC2', 'TC3', 'TOP'],
                  '17ft': ['TC1', 'TC6', 'TC2', 'TC3', 'TOP']}
SENSOR_COLORS  = {'TC1':'C0', 'TC6':'C1', 'TC2':'C2', 'TC3':'C3', 'TOP':'C4'}
SENSOR_LABELS  = {'TC1':'S1', 'TC6':'S6', 'TC2':'S2', 'TC3':'S3', 'TOP':'TOP'}

PRESSURE_SUBSAMPLE = 10   # every N timesteps for pressure extraction

# ============================================================
# TC radial correction (appendix check only)
# ============================================================
PIPE_ID_IN           = 4.0
TC_DIST_FROM_WALL_IN = 1.25
N_PROFILE            = 7
N_PROFILE_RANGE      = [5, 7, 10]
PIPE_R_IN  = PIPE_ID_IN / 2.0
TC_R_IN    = PIPE_R_IN - TC_DIST_FROM_WALL_IN
ETA        = TC_R_IN / PIPE_R_IN

# ============================================================
# Output directory
# ============================================================
TIMESTAMP  = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_ROOT   = REPO_ROOT / 'outputs' / 'paper_results_bundle' / TIMESTAMP
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Tests 17 ft: {TESTS_17FT}  |  rep: {REP_TEST_17FT}')
print(f'Tests 15 ft: {TESTS_15FT}  |  rep: {REP_TEST_15FT}')
print(f'A_leak = {A_LEAK_CM2} cm^2, TC1 offset = {TC1_OFFSET_C} C')
print(f'Cd = {CD}, K_stove = {K_STOVE}')
print(f'DT = {DT} s, max sim = {SIM_TIME_MAX} s')
print(f'Output -> {OUT_ROOT}')

## Helper functions

In [ ]:
# ============================================================
# Building blocks reused from results_validation
# ============================================================

def smooth_transition(t, t_start, t_end, val_start, val_end):
    """Cosine ramp between val_start and val_end over [t_start, t_end]."""
    if t <= t_start: return val_start
    if t >= t_end:   return val_end
    frac = (t - t_start) / (t_end - t_start)
    return val_start + (val_end - val_start) * 0.5 * (1.0 - np.cos(np.pi * frac))


def build_parts(chimney_type, T_indoor_K, T_outdoor_K):
    geom = el.get_chimney_geometry(chimney_type)
    bulk_vert_in = geom['final_vert_in'] - FINE_TOP_IN
    return [
        mat_props.create_single_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=BLACK_PIPE_IN/12,
            z_ft=0, material='steel_24_gauge', orientation='vertical',
            T_ambient_K=T_indoor_K, Pa=P_ATM, num_segments=6),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=HORIZ_IN/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='horizontal',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=2),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=bulk_vert_in/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='vertical',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=8),
        mat_props.create_triple_wall_chimney_part(
            inner_diameter_ft=DIAMETER_FT, height_ft=FINE_TOP_IN/12, z_ft=0,
            inner_material='stainless_430', middle_material='ceramic_fiber',
            outer_material='stainless_316', orientation='vertical',
            T_ambient_K=T_outdoor_K, Pa=P_ATM, num_segments=FINE_TOP_SEGS),
    ]


def build_fittings(T_indoor_K, T_outdoor_K):
    return [
        {'type': 'tee', 'between_parts': (-1, 0),
         'horizontal_length_ft': TEE1_HORIZ_IN/12,
         'vertical_length_ft':   TEE1_VERT_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'single',
         'material': 'steel_24_gauge', 'T_ambient_K': T_indoor_K},
        {'type': 'elbow', 'between_parts': (0, 1),
         'height_ft': ELBOW_VERT_IN/12, 'width_ft': ELBOW_HORIZ_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'single',
         'material': 'steel_24_gauge', 'T_ambient_K': T_indoor_K,
         'angle_deg': 90, 'R_over_D': 1.5, 'elbow_type': 'long_radius'},
        {'type': 'tee', 'between_parts': (1, 2),
         'horizontal_length_ft': TEE2_HORIZ_IN/12,
         'vertical_length_ft':   TEE2_VERT_IN/12,
         'diameter_ft': DIAMETER_FT, 'wall_type': 'triple',
         'inner_material': 'stainless_430', 'middle_material': 'ceramic_fiber',
         'outer_material': 'stainless_316', 'T_ambient_K': T_outdoor_K},
    ]


def make_door_func(test_id):
    cfg     = el.get_test_config(test_id)
    offsets = cfg['door_events']['offsets_s']
    door_in = cfg['door_events'].get('door_opening_in', 2.0)
    RAMP    = DOOR_RAMP_S
    cracked_pct = (
        np.degrees(np.arcsin(
            min(door_in, DOOR_HEIGHT_IN * 0.999) / DOOR_HEIGHT_IN
        )) / 90.0 * 100.0
    )
    door_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
    t_crack  = offsets[door_key]
    t_close  = offsets['door_closed']
    t_reopen = offsets.get('door_reopen_2in', None)
    t_close2 = offsets.get('door_closed_again', None)

    def door_func(t):
        if t < t_crack - RAMP:
            return 100.0
        if t < t_crack:
            return smooth_transition(t, t_crack - RAMP, t_crack, 100.0, cracked_pct)
        if t < t_close:
            return cracked_pct
        if t < t_close + RAMP:
            return smooth_transition(t, t_close, t_close + RAMP, cracked_pct, 0.0)
        if t_reopen is not None:
            if t < t_reopen:
                return 0.0
            if t < t_reopen + RAMP:
                return smooth_transition(t, t_reopen, t_reopen + RAMP, 0.0, cracked_pct)
            if t_close2 is not None and t < t_close2:
                return cracked_pct
            if t_close2 is not None and t < t_close2 + RAMP:
                return smooth_transition(t, t_close2, t_close2 + RAMP, cracked_pct, 0.0)
        return 0.0
    return door_func


def run_sim(test_id):
    """Run one simulation (rain_cap, no distributed leakage, relax_to_steady)."""
    cfg      = el.get_test_config(test_id)
    ct       = cfg['chimney_type']
    cond     = cfg['conditions']
    offsets  = cfg['door_events']['offsets_s']
    T_in_K   = cond['T_indoor_C']  + 273.15
    T_out_K  = cond['T_outdoor_C'] + 273.15
    wind_ms  = cond['wind_speed_m_s']

    parts    = build_parts(ct, T_in_K, T_out_K)
    fittings = build_fittings(T_in_K, T_out_K)
    door_fn  = make_door_func(test_id)

    tc1_data = el.get_temperature_data(test_id, 'TC1')
    tc1_K    = np.asarray(tc1_data['temperature_C'], dtype=float) + TC1_OFFSET_C + 273.15
    inlet_fn = interp1d(
        tc1_data['time_s'], tc1_K,
        kind='linear', bounds_error=False, fill_value='extrapolate'
    )

    sim_time = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)

    inlet_cfg = {
        'enable':          True,
        'door_type':       'swivel',
        'door_width_ft':   DOOR_WIDTH_IN  / 12,
        'door_height_ft':  DOOR_HEIGHT_IN / 12,
        'Cd':              CD,
        'A_leak_m2':       A_LEAK_CM2 * 1e-4,
        'inlet_area_min_m2': 0.0,
        'K_stove':         K_STOVE,
    }

    top_bnd = {
        'enable_wind_cap_effect': True,
        'cap_type':               'rain_cap',
        'wind_speed_m_s':         wind_ms,
    }

    ch = Chimney(
        parts=parts, elbows=fittings,
        dt=DT, total_time=sim_time,
        use_coupled_solver=True,
        door_opening_func=door_fn,
        inlet_temp_func=inlet_fn,
        inlet_restriction=inlet_cfg,
        top_boundary=top_bnd,
        leakage_config=None,      # EXPLICIT: no distributed leakage
        enable_diagnostics=True,
        verbose=False,
    )
    ch.run_simulation()
    return ch


def find_segment(ch, part_idx, height_in):
    target_m = ch.part_base_elevations[part_idx] + height_in * 0.0254
    return int(np.argmin(np.abs(ch.segment_elevations - target_m)))


def get_door_event_times(test_id):
    offsets = el.get_test_config(test_id)['door_events']['offsets_s']
    keys = ['door_to_2in', 'door_to_3in', 'door_closed',
            'door_reopen_2in', 'door_closed_again']
    return sorted(offsets[k] for k in keys if k in offsets)


def get_phase_windows(test_id):
    """Return (t_cracked_start, t_cracked_end, t_closed_start, t_closed_end).
    Same segmentation as results_validation RMSE cell."""
    offsets  = el.get_test_config(test_id)['door_events']['offsets_s']
    door_key = 'door_to_2in' if 'door_to_2in' in offsets else 'door_to_3in'
    t_crack  = offsets[door_key]
    t_close  = offsets['door_closed']
    t_end    = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)
    return (t_crack, t_close, t_close + 30, t_end)


def compute_rmse_bias(exp_t, exp_v, sim_t, sim_v, t_window):
    exp_t = np.asarray(exp_t, dtype=float)
    exp_v = np.asarray(exp_v, dtype=float)
    mask  = (exp_t >= t_window[0]) & (exp_t <= t_window[1])
    et, ev = exp_t[mask], exp_v[mask]
    if len(et) < 5:
        return np.nan, np.nan
    sv   = np.interp(et, sim_t, sim_v)
    diff = sv - ev
    return np.sqrt(np.nanmean(diff**2)), float(np.nanmean(diff))


def savefig(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  saved: {path.relative_to(REPO_ROOT)}')


def get_sim_wall_T(ch, seg):
    if hasattr(ch, 'T_wall') and ch.T_wall is not None:
        return ch.T_wall[:, seg]
    return ch.T_gas[:, seg]


def compute_tc_predicted(T_gas_K, T_wall_K, eta, n=7):
    T_gas_K  = np.asarray(T_gas_K,  dtype=float)
    T_wall_K = np.asarray(T_wall_K, dtype=float)
    T_pred   = T_wall_K + (T_gas_K - T_wall_K) * (1.0 - eta ** n)
    T_lo = np.minimum(T_gas_K, T_wall_K)
    T_hi = np.maximum(T_gas_K, T_wall_K)
    return np.clip(T_pred, T_lo, T_hi)


def get_pitot_mdot(test_id):
    dp_data  = el.get_sensor_data(test_id, 'pitot_dp', as_gauge=False)
    tc4_data = el.get_temperature_data(test_id, 'TC4')
    top_data = el.get_sensor_data(test_id, 'TOP', as_gauge=True)

    t_dp = np.asarray(dp_data['time_s'],      dtype=float)
    dp   = np.asarray(dp_data['pressure_Pa'],  dtype=float)
    t_tc4 = np.asarray(tc4_data['time_s'],        dtype=float)
    T_tc4 = np.asarray(tc4_data['temperature_C'], dtype=float) + 273.15
    t_top = np.asarray(top_data['time_s'],      dtype=float)
    P_top = np.asarray(top_data['pressure_Pa'], dtype=float)

    T_g = np.interp(t_dp, t_tc4, T_tc4, left=T_tc4[0],  right=T_tc4[-1])
    P_g = np.interp(t_dp, t_top, P_top, left=P_top[0],   right=P_top[-1])

    R_AIR = 287.0
    P_abs = P_ATM + P_g
    rho   = P_abs / (R_AIR * np.maximum(T_g, 200.0))
    dp_pos = np.maximum(dp, 0.0)
    v      = np.sqrt(2.0 * dp_pos / np.maximum(rho, 0.1))
    mdot   = rho * v * PIPE_AREA_M2

    dt_est  = max(float(np.median(np.diff(t_dp))), 0.1)
    win     = max(1, int(10.0 / dt_est))
    mdot_sm = uniform_filter1d(mdot, size=win)
    return t_dp, mdot * 1e3, mdot_sm * 1e3   # g/s


print('Helpers defined.')

## Load experimental data

In [ ]:
exp_data = {}

for test_id in TESTS_ALL:
    cfg = el.get_test_config(test_id)
    ct  = cfg['chimney_type']

    temp  = {s: el.get_temperature_data(test_id, s) for s in TEMP_SENSORS[ct]}
    press = {s: el.get_sensor_data(test_id, s, as_gauge=True)
             for s in PRESS_SENSORS[ct]}

    tc4   = el.get_temperature_data(test_id, 'TC4')
    pitot = el.get_sensor_data(test_id, 'pitot_dp', as_gauge=False)

    exp_data[test_id] = {
        'chimney_type': ct,
        'config':       cfg,
        'temperature':  temp,
        'pressure':     press,
        'tc4':          tc4,
        'pitot':        pitot,
    }
    cond = cfg['conditions']
    print(f'{test_id} ({ct}): wind={cond["wind_speed_m_s"]:.1f} m/s, '
          f'T_in={cond["T_indoor_C"]:.0f} C, T_out={cond["T_outdoor_C"]:.0f} C')

print(f'\nLoaded {len(exp_data)} tests.')

## Run simulations

In [ ]:
results  = {}
t_total0 = _time.time()

for test_id in TESTS_ALL:
    ct = exp_data[test_id]['chimney_type']
    t0 = _time.time()
    ch = run_sim(test_id)
    elapsed = _time.time() - t0
    results[test_id] = ch
    n_segs  = ch.T_gas.shape[1]
    n_steps = ch.T_gas.shape[0]
    print(f'  {test_id} ({ct}): {elapsed:.1f}s,  {n_segs} segs, {n_steps} steps')

print(f'\nAll {len(results)} sims done in {_time.time()-t_total0:.0f} s.')

In [ ]:
print('Leakage area verification (must all be 0.0 cm^2):')
for test_id, ch in results.items():
    total_cm2 = float(np.sum(ch.segment_leakage_area_m2) / 1e-4)
    ok = total_cm2 < 1e-9
    tag = 'OK' if ok else 'FAIL'
    print(f'  {test_id}: {total_cm2:.2e} cm^2  [{tag}]')
    if not ok:
        raise RuntimeError('Leakage check failed')

## Pre-compute sim quantities at sensor locations

In [ ]:
# -- Gauge pressure at sensor locations --
sim_pressure = {}   # test_id -> {sensor: (times_s, P_gauge_Pa)}

for test_id in TESTS_ALL:
    ct  = exp_data[test_id]['chimney_type']
    ch  = results[test_id]
    n   = ch.T_gas.shape[0]
    seg_map = {s: find_segment(ch, *SENSOR_MAP[ct][s]) for s in PRESS_SENSORS[ct]}
    step_indices = np.arange(0, n, PRESSURE_SUBSAMPLE)
    times        = step_indices * ch.dt
    unique_segs  = list(set(seg_map.values()))

    seg_P = {seg: np.zeros(len(step_indices)) for seg in unique_segs}
    for i, step in enumerate(step_indices):
        prof = ch.get_static_pressure_profile(timestep=int(step))
        for seg in unique_segs:
            seg_P[seg][i] = prof['P_gauge_Pa'][seg]

    sim_pressure[test_id] = {s: (times, seg_P[seg_map[s]]) for s in PRESS_SENSORS[ct]}

# -- TC-corrected simulated temperatures --
sim_tc_pred = {}   # test_id -> {sensor: {t, T_gas_C, T_wall_C, T_pred_C, shift_C}}

for test_id in TESTS_ALL:
    ct    = exp_data[test_id]['chimney_type']
    ch    = results[test_id]
    t_sim = np.arange(ch.T_gas.shape[0]) * ch.dt
    sim_tc_pred[test_id] = {}
    for sensor in TEMP_SENSORS[ct]:
        seg      = find_segment(ch, *SENSOR_MAP[ct][sensor])
        T_gas_K  = ch.T_gas[:, seg]
        T_wall_K = get_sim_wall_T(ch, seg)
        T_pred_K = compute_tc_predicted(T_gas_K, T_wall_K, ETA, N_PROFILE)
        sim_tc_pred[test_id][sensor] = {
            't':        t_sim,
            'T_gas_C':  T_gas_K  - 273.15,
            'T_wall_C': T_wall_K - 273.15,
            'T_pred_C': T_pred_K - 273.15,
            'shift_C':  (T_gas_K - T_pred_K),
        }

# -- Pitot-derived mass flow --
pitot_mdot = {}
for test_id in TESTS_ALL:
    pitot_mdot[test_id] = get_pitot_mdot(test_id)

# -- Pitot calibration JSON for exp steady-state values --
_pitot_json = REPO_ROOT / 'dev' / 'notebooks' / 'validation' / 'pitot_calibration_results.json'
with open(_pitot_json) as _f:
    pitot_cal = json.load(_f)

print('Pre-computation done: pressure, TC-corrected T, pitot mdot.')

## Compute steady-state quantities

In [ ]:
# Steady-state mdot (closed-door phase: [t_close+60, t_end])
ss_mdot_sim     = {}
ss_mdot_exp     = {}
ss_mdot_exp_std = {}

# Steady-state base draft (P_gauge at TC1, closed-door phase)
ss_draft_sim    = {}
ss_draft_exp    = {}

for test_id in TESTS_ALL:
    offsets = exp_data[test_id]['config']['door_events']['offsets_s']
    ch      = results[test_id]
    t_sim   = np.arange(ch.T_gas.shape[0]) * ch.dt
    t_close = offsets['door_closed']
    t_end   = min(offsets.get('test_stop', SIM_TIME_MAX), SIM_TIME_MAX)
    t_s, t_e = t_close + 60, t_end
    mask_ss = (t_sim >= t_s) & (t_sim <= t_e)

    # sim mdot
    mdot = ch.mass_flow_rate
    if mdot.ndim > 1:
        mdot = mdot[:, 0]
    ss_mdot_sim[test_id] = float(np.mean(mdot[mask_ss])) * 1000 if mask_ss.sum() > 0 else np.nan

    # exp mdot from pitot cal JSON
    if test_id in pitot_cal:
        ss_mdot_exp[test_id]     = pitot_cal[test_id]['experimental']['mdot_kgs'] * 1000
        ss_mdot_exp_std[test_id] = pitot_cal[test_id]['experimental']['mdot_std_kgs'] * 1000
    else:
        ss_mdot_exp[test_id]     = np.nan
        ss_mdot_exp_std[test_id] = np.nan

    # sim base draft at TC1
    t_p, P_tc1 = sim_pressure[test_id]['TC1']
    p_mask = (t_p >= t_s) & (t_p <= t_e)
    ss_draft_sim[test_id] = float(np.mean(P_tc1[p_mask])) if p_mask.sum() > 0 else np.nan

    # exp base draft at TC1
    pd_tc1 = exp_data[test_id]['pressure']['TC1']
    et = np.asarray(pd_tc1['time_s'],     dtype=float)
    ev = np.asarray(pd_tc1['pressure_Pa'], dtype=float)
    emask = (et >= t_s) & (et <= t_e)
    ss_draft_exp[test_id] = float(np.mean(ev[emask])) if emask.sum() > 0 else np.nan

print('Steady-state summary (closed-door, t > t_close+60):')
print(f'{"Test":<6} {"Type":<5} {"mdot_sim":>10} {"mdot_exp":>10} {"ratio":>7} '
      f'{"draft_sim":>10} {"draft_exp":>10}')
print('-' * 70)
for tid in TESTS_ALL:
    ct = exp_data[tid]['chimney_type']
    ratio = ss_mdot_sim[tid] / ss_mdot_exp[tid] if ss_mdot_exp[tid] > 0 else np.nan
    print(f'{tid:<6} {ct:<5} {ss_mdot_sim[tid]:>9.1f}  {ss_mdot_exp[tid]:>9.1f} '
          f'{ratio:>7.2f} {ss_draft_sim[tid]:>10.2f} {ss_draft_exp[tid]:>10.2f}')

## D1: Representative Temperature Overlays

In [ ]:
for test_id in [REP_TEST_17FT, REP_TEST_15FT]:
    ct         = exp_data[test_id]['chimney_type']
    sensors    = TEMP_SENSORS[ct]
    door_times = get_door_event_times(test_id)
    cond       = exp_data[test_id]['config']['conditions']
    ch         = results[test_id]
    t_sim      = np.arange(ch.T_gas.shape[0]) * ch.dt
    out_dir    = OUT_ROOT / test_id
    n_sensors  = len(sensors)

    # Door close time for vertical line
    t_close = exp_data[test_id]['config']['door_events']['offsets_s']['door_closed']

    fig, axes = plt.subplots(n_sensors, 1, figsize=(10, 3.2 * n_sensors),
                             sharex=True)
    if n_sensors == 1:
        axes = [axes]

    for ax, sensor in zip(axes, sensors):
        color = SENSOR_COLORS[sensor]
        lbl   = SENSOR_LABELS[sensor]
        td    = exp_data[test_id]['temperature'][sensor]
        t_exp = np.asarray(td['time_s'],       dtype=float)
        T_exp = np.asarray(td['temperature_C'], dtype=float)
        d     = sim_tc_pred[test_id][sensor]

        ax.plot(t_exp / 60, T_exp, '-', color=color, lw=0.9, alpha=0.8,
                label=f'{lbl} exp')
        ax.plot(d['t'] / 60, d['T_gas_C'], '--', color=color, lw=1.5,
                label=f'{lbl} sim')

        # Mark cracked/closed boundary
        ax.axvline(t_close / 60, color='k', ls=':', lw=1.0, alpha=0.6)
        ax.text(t_close / 60 + 0.15, ax.get_ylim()[0] + 0.92 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
                'door closed', fontsize=8, color='k', alpha=0.7,
                ha='left', va='top')

        for dt_s in door_times:
            if dt_s != t_close:
                ax.axvline(dt_s / 60, color='red', ls='--', lw=0.7, alpha=0.4)

        ax.set_ylabel(f'{lbl} T (C)', fontsize=11)
        ax.legend(fontsize=9, loc='lower right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)', fontsize=12)
    axes[-1].set_xlim(0, t_sim[-1] / 60)
    axes[0].set_title(
        f'Temperature: Test {test_id} ({ct})  '
        f'wind {cond["wind_speed_m_s"]:.1f} m/s, T_out {cond["T_outdoor_C"]:.0f} C',
        fontsize=12)

    savefig(fig, out_dir / 'fig_T_all.png')
    plt.show()
    plt.close(fig)

## D2: Representative Pressure Overlays

In [ ]:
for test_id in [REP_TEST_17FT, REP_TEST_15FT]:
    ct         = exp_data[test_id]['chimney_type']
    sensors    = PRESS_SENSORS[ct]
    door_times = get_door_event_times(test_id)
    cond       = exp_data[test_id]['config']['conditions']
    t_sim_max  = (results[test_id].T_gas.shape[0] - 1) * results[test_id].dt
    out_dir    = OUT_ROOT / test_id
    n_sensors  = len(sensors)

    t_close = exp_data[test_id]['config']['door_events']['offsets_s']['door_closed']

    fig, axes = plt.subplots(n_sensors, 1, figsize=(10, 3.0 * n_sensors),
                             sharex=True)
    if n_sensors == 1:
        axes = [axes]

    for ax, sensor in zip(axes, sensors):
        color = SENSOR_COLORS[sensor]
        lbl   = SENSOR_LABELS[sensor]
        pd_   = exp_data[test_id]['pressure'][sensor]
        t_p, P_g = sim_pressure[test_id][sensor]

        ax.plot(np.asarray(pd_['time_s']) / 60,
                np.asarray(pd_['pressure_Pa']),
                '-', color=color, lw=0.9, alpha=0.8,
                label=f'{lbl} exp')
        ax.plot(t_p / 60, P_g, '--', color=color, lw=1.5,
                label=f'{lbl} sim')

        ax.axhline(0, color='gray', ls=':', alpha=0.4)
        ax.axvline(t_close / 60, color='k', ls=':', lw=1.0, alpha=0.6)
        for dt_s in door_times:
            if dt_s != t_close:
                ax.axvline(dt_s / 60, color='red', ls='--', lw=0.7, alpha=0.4)

        ax.set_ylabel(f'{lbl} P (Pa)', fontsize=11)
        ax.legend(fontsize=9, loc='lower right')
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)', fontsize=12)
    axes[-1].set_xlim(0, t_sim_max / 60)
    axes[0].set_title(
        f'Gauge Pressure: Test {test_id} ({ct})  '
        f'wind {cond["wind_speed_m_s"]:.1f} m/s, T_out {cond["T_outdoor_C"]:.0f} C',
        fontsize=12)

    savefig(fig, out_dir / 'fig_P_all.png')
    plt.show()
    plt.close(fig)

## D3: RMSE / Bias Summary Plots

In [ ]:
# Compute per-test, per-sensor RMSE/bias for cracked + closed windows
rmse_T_rows = []
rmse_P_rows = []

for test_id in TESTS_ALL:
    ct = exp_data[test_id]['chimney_type']
    t_crk0, t_crk1, t_cls0, t_cls1 = get_phase_windows(test_id)
    ch    = results[test_id]
    sim_t = np.arange(ch.T_gas.shape[0]) * ch.dt
    height_group = '17 ft' if ct == '17ft' else '15 ft'

    # Temperature
    for sensor in TEMP_SENSORS[ct]:
        d     = sim_tc_pred[test_id][sensor]
        sim_T = d['T_gas_C']
        td    = exp_data[test_id]['temperature'][sensor]
        et    = np.asarray(td['time_s'],       dtype=float)
        ev    = np.asarray(td['temperature_C'], dtype=float)

        rmse_c, bias_c = compute_rmse_bias(et, ev, sim_t, sim_T, (t_crk0, t_crk1))
        rmse_d, bias_d = compute_rmse_bias(et, ev, sim_t, sim_T, (t_cls0, t_cls1))

        rmse_T_rows.append({
            'Test': test_id, 'Height': height_group, 'Sensor': sensor,
            'RMSE_cracked': rmse_c, 'Bias_cracked': bias_c,
            'RMSE_closed':  rmse_d, 'Bias_closed':  bias_d,
        })

    # Pressure
    for sensor in PRESS_SENSORS[ct]:
        t_p, P_sim = sim_pressure[test_id][sensor]
        pd_ = exp_data[test_id]['pressure'][sensor]
        et  = np.asarray(pd_['time_s'],     dtype=float)
        ev  = np.asarray(pd_['pressure_Pa'], dtype=float)

        rmse_c, bias_c = compute_rmse_bias(et, ev, t_p, P_sim, (t_crk0, t_crk1))
        rmse_d, bias_d = compute_rmse_bias(et, ev, t_p, P_sim, (t_cls0, t_cls1))

        rmse_P_rows.append({
            'Test': test_id, 'Height': height_group, 'Sensor': sensor,
            'RMSE_cracked': rmse_c, 'Bias_cracked': bias_c,
            'RMSE_closed':  rmse_d, 'Bias_closed':  bias_d,
        })

df_T = pd.DataFrame(rmse_T_rows)
df_P = pd.DataFrame(rmse_P_rows)

print('Temperature RMSE/Bias (C):')
print(df_T.to_string(index=False, float_format='%.1f'))
print()
print('Pressure RMSE/Bias (Pa):')
print(df_P.to_string(index=False, float_format='%.2f'))

In [ ]:
# Temperature RMSE summary: grouped bar chart (cracked vs closed), by sensor
# Group: height_group + sensor
g = df_T.groupby(['Height', 'Sensor']).agg(
    RMSE_cracked=('RMSE_cracked', 'mean'),
    RMSE_closed=('RMSE_closed', 'mean'),
    Bias_cracked=('Bias_cracked', 'mean'),
    Bias_closed=('Bias_closed', 'mean'),
).reset_index()
g['label'] = g['Height'] + ' ' + g['Sensor']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(len(g))
w = 0.35
ax1.bar(x - w/2, g['RMSE_cracked'], w, label='Cracked', color='C0', alpha=0.85)
ax1.bar(x + w/2, g['RMSE_closed'],  w, label='Closed',  color='C1', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(g['label'], fontsize=9, rotation=30, ha='right')
ax1.set_ylabel('RMSE (C)', fontsize=12)
ax1.set_title('Temperature RMSE by sensor (avg across tests)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x - w/2, g['Bias_cracked'], w, label='Cracked', color='C0', alpha=0.85)
ax2.bar(x + w/2, g['Bias_closed'],  w, label='Closed',  color='C1', alpha=0.85)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(g['label'], fontsize=9, rotation=30, ha='right')
ax2.set_ylabel('Bias (C)  [sim - exp]', fontsize=12)
ax2.set_title('Temperature Bias by sensor (avg across tests)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

savefig(fig, OUT_ROOT / 'fig_T_RMSE_summary.png')
plt.show()
plt.close(fig)

In [ ]:
# Pressure RMSE summary: grouped bar chart (cracked vs closed), by sensor
gp = df_P.groupby(['Height', 'Sensor']).agg(
    RMSE_cracked=('RMSE_cracked', 'mean'),
    RMSE_closed=('RMSE_closed', 'mean'),
    Bias_cracked=('Bias_cracked', 'mean'),
    Bias_closed=('Bias_closed', 'mean'),
).reset_index()
gp['label'] = gp['Height'] + ' ' + gp['Sensor']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(gp))
w = 0.35
ax1.bar(x - w/2, gp['RMSE_cracked'], w, label='Cracked', color='C0', alpha=0.85)
ax1.bar(x + w/2, gp['RMSE_closed'],  w, label='Closed',  color='C1', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(gp['label'], fontsize=9, rotation=30, ha='right')
ax1.set_ylabel('RMSE (Pa)', fontsize=12)
ax1.set_title('Pressure RMSE by sensor (avg across tests)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x - w/2, gp['Bias_cracked'], w, label='Cracked', color='C0', alpha=0.85)
ax2.bar(x + w/2, gp['Bias_closed'],  w, label='Closed',  color='C1', alpha=0.85)
ax2.axhline(0, color='k', lw=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(gp['label'], fontsize=9, rotation=30, ha='right')
ax2.set_ylabel('Bias (Pa)  [sim - exp]', fontsize=12)
ax2.set_title('Pressure Bias by sensor (avg across tests)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

savefig(fig, OUT_ROOT / 'fig_P_RMSE_summary.png')
plt.show()
plt.close(fig)

## D4: Mass Flow

In [ ]:
# Per-test mdot time series
for test_id in TESTS_ALL:
    ct         = exp_data[test_id]['chimney_type']
    door_times = get_door_event_times(test_id)
    cond       = exp_data[test_id]['config']['conditions']
    ch         = results[test_id]
    t_sim      = np.arange(ch.T_gas.shape[0]) * ch.dt
    out_dir    = OUT_ROOT / test_id

    t_close = exp_data[test_id]['config']['door_events']['offsets_s']['door_closed']

    mdot_sim = ch.mass_flow_rate
    if mdot_sim.ndim > 1:
        mdot_sim = mdot_sim[:, 0]
    mdot_sim_gs = mdot_sim * 1000

    t_pit, mdot_pit_raw, mdot_pit_sm = pitot_mdot[test_id]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t_sim / 60, mdot_sim_gs, 'C0-', lw=1.5, label='sim')
    ax.plot(t_pit / 60, mdot_pit_sm, 'C1-', lw=1.0, alpha=0.85,
            label='exp pitot (10 s avg)')
    ax.fill_between(t_pit / 60, 0, mdot_pit_raw, color='C1', alpha=0.12)

    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.axvline(t_close / 60, color='k', ls=':', lw=1.0, alpha=0.6)
    for dt_s in door_times:
        if dt_s != t_close:
            ax.axvline(dt_s / 60, color='red', ls='--', lw=0.7, alpha=0.4)

    if not np.isnan(ss_mdot_sim[test_id]):
        ax.axhline(ss_mdot_sim[test_id], color='C0', ls=':', alpha=0.6, lw=1.2)
    if not np.isnan(ss_mdot_exp[test_id]):
        ax.axhline(ss_mdot_exp[test_id], color='C1', ls=':', alpha=0.6, lw=1.2)

    ax.set_xlim(0, t_sim[-1] / 60)
    ax.set_xlabel('Time (min)', fontsize=12)
    ax.set_ylabel('Mass flow (g/s)', fontsize=12)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3)
    ratio = ss_mdot_sim[test_id] / ss_mdot_exp[test_id] \
            if not np.isnan(ss_mdot_exp[test_id]) else np.nan
    ax.set_title(
        f'Mass Flow: {test_id} ({ct})  '
        f'sim={ss_mdot_sim[test_id]:.1f}, exp={ss_mdot_exp[test_id]:.1f} g/s '
        f'(ratio {ratio:.2f}x)', fontsize=11)

    savefig(fig, out_dir / 'fig_mdot_timeseries.png')
    plt.show()
    plt.close(fig)

In [ ]:
# Steady-state mdot comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
x  = np.arange(len(TESTS_ALL))
w  = 0.35
labels_x = [f'{tid}\n({exp_data[tid]["chimney_type"]})' for tid in TESTS_ALL]

ax.bar(x - w / 2,
       [ss_mdot_sim[t] for t in TESTS_ALL],
       w, label='Sim (SS)', color='C0', alpha=0.85)
ax.bar(x + w / 2,
       [ss_mdot_exp[t] for t in TESTS_ALL],
       w,
       yerr=[ss_mdot_exp_std[t] for t in TESTS_ALL],
       label='Exp pitot (1-sigma)', color='C1', alpha=0.85,
       capsize=5, error_kw={'lw': 1.5})

ax.set_xticks(x)
ax.set_xticklabels(labels_x, fontsize=11)
ax.set_ylabel('Mass flow rate (g/s)', fontsize=13)
ax.set_ylim(0, max(ss_mdot_sim.values()) * 1.35)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4)
ax.set_title('Steady-state mass flow: sim vs. pitot experiment', fontsize=12)

for i, tid in enumerate(TESTS_ALL):
    if not np.isnan(ss_mdot_exp[tid]) and ss_mdot_exp[tid] > 0:
        ratio = ss_mdot_sim[tid] / ss_mdot_exp[tid]
        y_top = max(ss_mdot_sim[tid], ss_mdot_exp[tid] + ss_mdot_exp_std[tid]) + 0.5
        ax.text(i, y_top, f'{ratio:.2f}x',
                ha='center', va='bottom', fontsize=9, color='dimgray')

savefig(fig, OUT_ROOT / 'fig_mdot_ss_comparison.png')
plt.show()
plt.close(fig)

## D5: Height Scaling (15 ft vs 17 ft)

In [ ]:
# Collect height-grouped averages
groups = [
    ('17 ft', 17, TESTS_17FT),
    ('15 ft', 15, TESTS_15FT),
]

heights_ft   = []
mdot_sim_avg = []
mdot_exp_avg = []
mdot_exp_err = []
draft_sim_avg = []
draft_exp_avg = []

for label, h, tids in groups:
    heights_ft.append(h)
    mdot_sim_avg.append(np.mean([ss_mdot_sim[t] for t in tids]))
    mdot_exp_avg.append(np.mean([ss_mdot_exp[t] for t in tids]))
    # propagate uncertainty: std of means (combine per-test stds)
    mdot_exp_err.append(np.sqrt(np.mean([ss_mdot_exp_std[t]**2 for t in tids])))
    draft_sim_avg.append(np.mean([ss_draft_sim[t] for t in tids]))
    draft_exp_avg.append(np.mean([ss_draft_exp[t] for t in tids]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

# Left: mdot vs height
ax1.plot(heights_ft, mdot_sim_avg, 's-', color='C0', ms=8, lw=1.5, label='Sim')
ax1.errorbar(heights_ft, mdot_exp_avg, yerr=mdot_exp_err,
             fmt='o-', color='C1', ms=8, lw=1.5, capsize=5, label='Exp pitot')
# Also show individual tests
for label, h, tids in groups:
    for t in tids:
        ax1.plot(h, ss_mdot_sim[t], 's', color='C0', ms=5, alpha=0.4)
        ax1.plot(h, ss_mdot_exp[t], 'o', color='C1', ms=5, alpha=0.4)

ax1.set_xlabel('Chimney height (ft)', fontsize=12)
ax1.set_ylabel('SS mass flow (g/s)', fontsize=12)
ax1.set_title('Mass flow vs. chimney height', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(heights_ft)

# Right: base draft vs height
ax2.plot(heights_ft, draft_sim_avg, 's-', color='C0', ms=8, lw=1.5, label='Sim')
ax2.plot(heights_ft, draft_exp_avg, 'o-', color='C1', ms=8, lw=1.5, label='Exp')
for label, h, tids in groups:
    for t in tids:
        ax2.plot(h, ss_draft_sim[t], 's', color='C0', ms=5, alpha=0.4)
        ax2.plot(h, ss_draft_exp[t], 'o', color='C1', ms=5, alpha=0.4)

ax2.set_xlabel('Chimney height (ft)', fontsize=12)
ax2.set_ylabel('Base draft at S1 (Pa)', fontsize=12)
ax2.set_title('Base draft vs. chimney height', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(heights_ft)

savefig(fig, OUT_ROOT / 'fig_height_scaling.png')
plt.show()
plt.close(fig)

## D6: TC Radial Correction Check (Appendix)

In [ ]:
print('TC Radial Correction Summary')
print(f'  Pipe ID     = {PIPE_ID_IN} in  =>  R = {PIPE_R_IN} in')
print(f'  TC from wall = {TC_DIST_FROM_WALL_IN} in  =>  r_TC = {TC_R_IN} in from centerline')
print(f'  eta = r_TC / R = {ETA:.4f}')
print()
for _n in N_PROFILE_RANGE:
    _sf = ETA ** _n
    print(f'  n={_n}: eta^n = {_sf:.6f}  =>  (1 - eta^n) = {1-_sf:.6f}')
print()

# Compute typical DeltaT range across all tests/sensors in closed-door SS
shifts = []
for test_id in TESTS_ALL:
    ct = exp_data[test_id]['chimney_type']
    t_crk0, t_crk1, t_cls0, t_cls1 = get_phase_windows(test_id)
    for sensor in TEMP_SENSORS[ct]:
        d = sim_tc_pred[test_id][sensor]
        mask = (d['t'] >= t_cls0) & (d['t'] <= t_cls1)
        if mask.sum() > 0:
            shifts.append(float(np.mean(d['shift_C'][mask])))

print(f'Steady-state DeltaT_shift (T_gas - T_TC_pred):')
print(f'  min = {min(shifts):.4f} C')
print(f'  max = {max(shifts):.4f} C')
print(f'  => Correction is negligible for this geometry.')

## Report

In [ ]:
# Build compact markdown report
_params = (
    f'A_leak = {A_LEAK_CM2} cm^2 (inlet restriction), leakage_config = None\n'
    f'TC1_offset = {TC1_OFFSET_C} C, Cd = {CD}, K_stove = {K_STOVE}\n'
    f'Door = swivel {DOOR_WIDTH_IN}" x {DOOR_HEIGHT_IN}", ramp = {DOOR_RAMP_S} s\n'
    f'Cap = rain_cap (Cp = -0.30), P_ATM = {P_ATM} Pa\n'
    f'Solver = relax_to_steady, dt = {DT} s, max_time = {SIM_TIME_MAX} s\n'
    f'Branch = paper/results (no BE-CV, no leakage-enthalpy, no state-consistency)\n'
    f'TC correction: eta={ETA:.4f}, n={N_PROFILE}, shift_factor={ETA**N_PROFILE:.6f} (negligible)'
)

_leak_check = '\n'.join(
    f'{tid}: sum(seg_leak_area) = '
    f'{float(np.sum(results[tid].segment_leakage_area_m2)/1e-4):.2e} cm^2'
    for tid in TESTS_ALL
)

# mdot table
_mdot_rows = []
_ratios = []
for tid in TESTS_ALL:
    ratio = ss_mdot_sim[tid] / ss_mdot_exp[tid] \
            if not np.isnan(ss_mdot_exp[tid]) and ss_mdot_exp[tid] > 0 else np.nan
    _ratios.append(ratio)
    _mdot_rows.append({
        'Test': tid,
        'Type': exp_data[tid]['chimney_type'],
        'mdot_sim_gs': round(ss_mdot_sim[tid], 1),
        'mdot_exp_gs': round(ss_mdot_exp[tid], 1),
        'mdot_std_gs': round(ss_mdot_exp_std[tid], 1),
        'sim_exp_ratio': round(ratio, 2),
    })
df_mdot = pd.DataFrame(_mdot_rows)
_valid_ratios = [r for r in _ratios if not np.isnan(r)]
_ratio_str = (f'Sim over-prediction: {min(_valid_ratios):.2f}-{max(_valid_ratios):.2f}x '
              f'(mean {np.mean(_valid_ratios):.2f}x)'
              if _valid_ratios else 'No exp data.')

# height scaling table
_hs_rows = []
for label, h, tids in [('17 ft', 17, TESTS_17FT), ('15 ft', 15, TESTS_15FT)]:
    _hs_rows.append({
        'Height': label,
        'mdot_sim_gs': round(np.mean([ss_mdot_sim[t] for t in tids]), 1),
        'mdot_exp_gs': round(np.mean([ss_mdot_exp[t] for t in tids]), 1),
        'draft_sim_Pa': round(np.mean([ss_draft_sim[t] for t in tids]), 2),
        'draft_exp_Pa': round(np.mean([ss_draft_exp[t] for t in tids]), 2),
    })
df_hs = pd.DataFrame(_hs_rows)

# figure list
_file_list = []
for tid in [REP_TEST_17FT, REP_TEST_15FT]:
    _file_list.append(f'{tid}/fig_T_all.png')
    _file_list.append(f'{tid}/fig_P_all.png')
for tid in TESTS_ALL:
    _file_list.append(f'{tid}/fig_mdot_timeseries.png')
_file_list += [
    'fig_T_RMSE_summary.png',
    'fig_P_RMSE_summary.png',
    'fig_mdot_ss_comparison.png',
    'fig_height_scaling.png',
]
_files_str = '\n'.join(f'  outputs/paper_results_bundle/{TIMESTAMP}/{f}' for f in _file_list)

report = f"""
-----BEGIN REPORT-----
# Paper Results Report
Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
Branch: paper/results
Tests: {', '.join(TESTS_ALL)}

## Parameters
{_params}

## Leakage verification (all must be 0 cm^2)
{_leak_check}

## Temperature RMSE / Bias [C]  (T_exp vs T_gas_sim)
{df_T.to_markdown(index=False, floatfmt='.1f')}

## Pressure RMSE / Bias [Pa]
{df_P.to_markdown(index=False, floatfmt='.2f')}

## Steady-state mass flow (sim vs pitot)
{df_mdot.to_markdown(index=False)}
  {_ratio_str}

## Height scaling (15 ft vs 17 ft) -- closed-door SS averages
{df_hs.to_markdown(index=False)}

## Saved figures
{_files_str}
-----END REPORT-----
"""

print(report)